# Ekstraksi Fitur Citra Sepatu: Rapi vs Berantakan
Jalankan `grayscale-converter.ipynb` terlebih dahulu.

- Fitur warna (dari `data/`): mean, std, skewness tiap kanal R, G, B
- Fitur GLCM (dari `grayscale/`): energi, entropi, kontras, korelasi, homogenitas
- Label: nama subfolder (`rapi`, `berantakan`)

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops

FOLDER_DATA = "data"
FOLDER_GRAY = "grayscale"
FOLDER_HASIL = "hasil"
KELAS = ["rapi", "berantakan"]
EKSTENSI = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
os.makedirs(FOLDER_HASIL, exist_ok=True)

def baca(path, flag=cv2.IMREAD_COLOR):
    img = cv2.imread(path, flag)
    if img is None:  # fallback path Windows
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), flag)
    return img

## 1. Membaca Data

In [2]:
df = pd.DataFrame([{"nama_file": n, "label": k}
                   for k in KELAS
                   for n in sorted(os.listdir(os.path.join(FOLDER_DATA, k)))
                   if n.lower().endswith(EKSTENSI)])
print("Total citra:", len(df))
print(df["label"].value_counts())

Total citra: 173
label
berantakan    97
rapi          76
Name: count, dtype: int64


## 2. Fitur Warna: Mean, Std, Skewness Tiap Kanal

In [3]:
warna = []
for _, row in df.iterrows():
    rgb = cv2.cvtColor(baca(os.path.join(FOLDER_DATA, row["label"], row["nama_file"])), cv2.COLOR_BGR2RGB)
    piksel = rgb.reshape(-1, 3).astype(np.float64)
    m, s, k = piksel.mean(axis=0), piksel.std(axis=0), skew(piksel, axis=0)
    warna.append({"mean_r": m[0], "mean_g": m[1], "mean_b": m[2],
                  "std_r": s[0], "std_g": s[1], "std_b": s[2],
                  "skew_r": k[0], "skew_g": k[1], "skew_b": k[2]})

fitur_warna = pd.concat([df[["nama_file"]], pd.DataFrame(warna), df[["label"]]], axis=1)
fitur_warna.to_csv(os.path.join(FOLDER_HASIL, "fitur_warna.csv"), index=False)
fitur_warna.head()

,nama_file,mean_r,mean_g,mean_b,std_r,std_g,std_b,skew_r,skew_g,skew_b,label
0,24-003 Rapi 2.jpg,167.528328,145.622662,129.198171,60.723361,60.684163,61.227238,-0.736099,-0.341286,-0.148620,rapi
1,24-003 Rapi 3.jpg,151.799273,144.180759,136.959967,56.835674,64.174024,71.231432,-0.638414,-0.310449,-0.249927,rapi
2,24-003 Rapi.jpeg,112.611609,105.590071,102.377592,51.803115,57.168554,63.539342,0.056719,0.195343,0.264629,rapi
3,24-006 Rapi 1.jpg,99.666263,84.972929,82.132553,63.248058,61.874065,63.158273,0.316514,0.817848,0.882425,rapi
4,24-006 Rapi 2.jpg,111.918504,102.250117,99.281981,65.353938,68.223014,69.404002,-0.013277,0.232739,0.281711,rapi


## 3. Matriks Co-occurrence (GLCM)
Contoh citra 3×3 dari modul: d = 1, θ = 0°, symmetric=False

In [4]:
contoh = np.array([[0, 1, 1],
                   [0, 2, 1],
                   [2, 2, 0]], dtype=np.uint8)

mentah = graycomatrix(contoh, distances=[1], angles=[0], levels=3, symmetric=False, normed=False)
norm   = graycomatrix(contoh, distances=[1], angles=[0], levels=3, symmetric=False, normed=True)
print("GLCM mentah:\n", mentah[:, :, 0, 0], "\nTotal pasangan:", mentah.sum())
print("\nGLCM ternormalisasi P(i,j):\n", np.round(norm[:, :, 0, 0], 3))

GLCM mentah:
 [[0 1 1]
 [0 1 0]
 [1 1 1]] 
Total pasangan: 6

GLCM ternormalisasi P(i,j):
 [[0.    0.167 0.167]
 [0.    0.167 0.   ]
 [0.167 0.167 0.167]]


## 4. Fitur GLCM: Energi, Entropi, Kontras, Korelasi, Homogenitas

In [5]:
def fitur_glcm(gray, levels=256):
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=levels, symmetric=False, normed=True)
    P = glcm[:, :, 0, 0]

    entropi = 0.0
    for p in P.ravel():
        if p > 0:                       # 0 * log2(0) = 0
            entropi -= p * np.log2(p)

    return {"energi":      graycoprops(glcm, "energy")[0, 0],
            "entropi":     entropi,
            "kontras":     graycoprops(glcm, "contrast")[0, 0],
            "korelasi":    graycoprops(glcm, "correlation")[0, 0],
            "homogenitas": graycoprops(glcm, "homogeneity")[0, 0]}

print("Contoh 3x3:", {k: round(float(v), 4) for k, v in fitur_glcm(contoh, levels=3).items()})

Contoh 3x3: {'energi': 0.4082, 'entropi': 2.585, 'kontras': 1.6667, 'korelasi': -0.3153, 'homogenitas': 0.5667}


In [6]:
def baca_gray(row):
    path = os.path.join(FOLDER_GRAY, row["label"], os.path.splitext(row["nama_file"])[0] + ".png")
    return baca(path, cv2.IMREAD_GRAYSCALE)

gray = baca_gray(df.iloc[0])
P = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=False, normed=True)[:, :, 0, 0]
print(df.iloc[0]["nama_file"], "| ukuran GLCM:", P.shape, "| jumlah P(i,j):", round(P.sum(), 4))
print("Fitur:", {k: round(float(v), 4) for k, v in fitur_glcm(gray).items()})

24-003 Rapi 2.jpg | ukuran GLCM: (256, 256) | jumlah P(i,j): 1.0
Fitur: {'energi': 0.0278, 'entropi': 11.934, 'kontras': 84.1391, 'korelasi': 0.988, 'homogenitas': 0.3766}


## 5. Tabel Fitur Lengkap

In [7]:
tekstur = pd.DataFrame([fitur_glcm(baca_gray(row)) for _, row in df.iterrows()])

fitur = pd.concat([fitur_warna.drop(columns="label"), tekstur, df[["label"]]], axis=1)
fitur.to_csv(os.path.join(FOLDER_HASIL, "fitur_lengkap.csv"), index=False)
print("Jumlah data:", len(fitur), "| jumlah fitur:", fitur.shape[1] - 2)
fitur

Jumlah data: 173 | jumlah fitur: 14


,nama_file,mean_r,mean_g,mean_b,std_r,std_g,std_b,skew_r,skew_g,skew_b,energi,entropi,kontras,korelasi,homogenitas,label
0,24-003 Rapi 2.jpg,167.528328,145.622662,129.198171,60.723361,60.684163,61.227238,-0.736099,-0.341286,-0.148620,0.027827,11.933976,84.139107,0.987996,0.376598,rapi
1,24-003 Rapi 3.jpg,151.799273,144.180759,136.959967,56.835674,64.174024,71.231432,-0.638414,-0.310449,-0.249927,0.027310,11.795616,100.085201,0.986491,0.436773,rapi
2,24-003 Rapi.jpeg,112.611609,105.590071,102.377592,51.803115,57.168554,63.539342,0.056719,0.195343,0.264629,0.025514,12.011990,114.308217,0.981392,0.378060,rapi
3,24-006 Rapi 1.jpg,99.666263,84.972929,82.132553,63.248058,61.874065,63.158273,0.316514,0.817848,0.882425,0.035975,10.979729,53.040742,0.992903,0.513641,rapi
4,24-006 Rapi 2.jpg,111.918504,102.250117,99.281981,65.353938,68.223014,69.404002,-0.013277,0.232739,0.281711,0.034417,10.951532,64.174080,0.992842,0.518268,rapi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,24-202 berantakan 2.jpeg,162.515012,155.952607,151.194528,50.304362,50.968246,50.344348,-0.962279,-0.722642,-0.609846,0.049552,10.671662,73.013444,0.985538,0.561912,berantakan
169,24-202 berantakan 3.jpeg,151.279449,140.899201,137.289909,41.720299,53.577764,58.204038,-0.833413,-0.479026,-0.367996,0.062526,10.100723,59.960283,0.987657,0.621249,berantakan
170,24-228 Berantakan 1.jpeg,141.339219,135.531790,132.263520,56.185976,61.387283,67.645255,-0.481903,-0.345294,-0.294260,0.089659,9.614470,30.943412,0.995638,0.652458,berantakan
171,24-228 Berantakan 2.jpeg,131.960992,133.630037,134.079376,53.359132,56.222276,60.655706,-0.660687,-0.443042,-0.127392,0.048706,10.481353,32.391536,0.994587,0.604647,berantakan
